In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "validation").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from validation.notebook_bootstrap import bootstrap, workshop_retrieved_references

bedrock_model_arn, load_workshop_state, persist_workshop_state_file = bootstrap()


# Passo a passo de customização de metadata em CSV
Este notebook fornece um código de exemplo passo a passo para o recurso 'CSV metadata customization', uma funcionalidade do Amazon Bedrock Knowledge Bases que aprimora o processamento de arquivos .csv separando conteúdo e metadata.

Para mais detalhes sobre esse recurso, leia este [blog](https://aws.amazon.com/blogs/machine-learning/knowledge-bases-for-amazon-bedrock-now-supports-advanced-parsing-chunking-and-query-reformulation-giving-greater-control-of-accuracy-in-rag-based-applications/#:~:text=Machine%20Learning%20Blog-,Knowledge%20Bases%20for%20Amazon%20Bedrock%20now%20supports%20advanced%20parsing%2C%20chunking,accuracy%20in%20RAG%20based%20applications).

## 1. Importar as bibliotecas necessárias
O primeiro passo é instalar os pacotes de pré-requisitos.

In [ ]:
%pip install --upgrade pip --quiet
%pip install -r ../requirements.txt --no-deps --quiet
%pip install -r ../requirements.txt --upgrade --quiet

In [ ]:
# Kernel restart is intentionally skipped in corrected notebooks.


In [ ]:
import botocore
botocore.__version__

Este código faz parte da configuração e é usado para:
- Adicionar o diretório pai ao system path do Python
- Importar um módulo customizado (BedrockStructuredKnowledgeBase) de `utils` necessário para execuções posteriores

In [ ]:
import sys
import logging
from pathlib import Path
import os
import time
import uuid
import boto3
import pprint
import json

current_path = Path().resolve()
current_path = current_path.parent

if str(current_path) not in sys.path:
    sys.path.append(str(current_path))

# Print sys.path to verify
print(sys.path)

from utils.knowledge_base import BedrockKnowledgeBase

In [ ]:
#Clients
s3_client = boto3.client('s3')
sts_client = boto3.client('sts')
session = boto3.session.Session()
region =  session.region_name
account_id = sts_client.get_caller_identity()["Account"]
bedrock_agent_client = boto3.client('bedrock-agent')
bedrock_agent_runtime_client = boto3.client('bedrock-agent-runtime') 
logging.basicConfig(format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)
region, account_id

In [ ]:
import time

# Get the current timestamp
current_time = time.time()

# Format the timestamp and add a UUID so the S3 bucket name is globally unique.
timestamp_str = time.strftime("%Y%m%d%H%M%S", time.localtime(current_time))
suffix = f"{timestamp_str}-{uuid.uuid4().hex}"
knowledge_base_name_standard = 'csv-metadata-kb'
knowledge_base_name_hierarchical = 'hierarchical-kb'
knowledge_base_description = "Knowledge Base csv metadata customization."
bucket_name = f'{knowledge_base_name_standard}-{suffix}'
foundation_model = os.getenv("BEDROCK_TEXT_MODEL_ID", "us.anthropic.claude-haiku-4-5-20251001-v1:0")

# Define data sources
data_source=[{"type": "S3", "bucket_name": bucket_name}]

## 2 - Criar knowledge bases com estratégia de fixed chunking
Vamos começar criando uma [Amazon Bedrock Knowledge Bases](https://aws.amazon.com/bedrock/knowledge-bases/) para armazenar dados de video games em formato csv. Knowledge Bases permitem integração com diferentes bancos de dados vetoriais incluindo [Amazon OpenSearch Serverless](https://aws.amazon.com/opensearch-service/features/serverless/), [Amazon Aurora](https://aws.amazon.com/rds/aurora/), [Pinecone](http://app.pinecone.io/bedrock-integration), [Redis Enterprise]() e [MongoDB Atlas](). Para este exemplo, vamos integrar a knowledge base com o Amazon OpenSearch Serverless. Para isso, usaremos a classe auxiliar `BedrockKnowledgeBase` que criará a knowledge base e todos os seus pré-requisitos:
1. IAM roles e policies
2. Bucket S3
3. Políticas de encryption, network e data access do Amazon OpenSearch Serverless
4. Collection do Amazon OpenSearch Serverless
5. Índice vetorial do Amazon OpenSearch Serverless
6. Knowledge base
7. Data source da Knowledge base

Vamos criar uma knowledge base usando a estratégia de fixed chunking.

Você pode escolher diferentes estratégias de chunking alterando os valores dos parâmetros abaixo:
```
"chunkingStrategy": "FIXED_SIZE | NONE | HIERARCHICAL | SEMANTIC"
```

In [ ]:
knowledge_base_standard = BedrockKnowledgeBase(
    kb_name=f'{knowledge_base_name_standard}-{suffix}',
    kb_description=knowledge_base_description,
    data_sources=data_source, 
    chunking_strategy = "FIXED_SIZE", 
    suffix = suffix
)

# Keep resources from this notebook discoverable by the workshop cleanup.
s3_client.put_bucket_tagging(
    Bucket=bucket_name,
    Tagging={"TagSet": [{"Key": "workshop-kb", "Value": "true"}]},
)

### 2.1 Baixar o dataset csv e fazer upload para o Amazon S3
Agora que criamos a knowledge base, vamos populá-la com o dataset `video_games.csv`. Esses dados estão sendo baixados [daqui](https://github.com/ali-ce/datasets/blob/master/Most-Expensive-Things/Videogames.csv). O dataset contém dados de vendas de video games originalmente coletados por Alice Corona e está licenciado sob uma [Creative Commons Attribution-ShareAlike 4.0 International License](https://github.com/ali-ce/datasets/blob/master/README.md#:~:text=Creative%20Commons%20Attribution%2DShareAlike%204.0%20International%20License.).


A data source da Knowledge Base espera que os dados estejam disponíveis no bucket S3 conectado a ela e alterações nos dados podem ser sincronizadas com a knowledge base usando a chamada de API `StartIngestionJob`. Neste exemplo usaremos a [abstração boto3](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent/client/start_ingestion_job.html) da API, por meio de nossas classes auxiliares.

In [ ]:
Path("csv_data").mkdir(parents=True, exist_ok=True)

In [ ]:
import csv
import io
from time import sleep
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


def download_and_validate_csv(url, filename, attempts=4, timeout=60):
    destination = Path(filename)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".part")
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            request = Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urlopen(request, timeout=timeout) as response:
                status = getattr(response, "status", 200)
                if status >= 400:
                    raise RuntimeError(f"HTTP {status} while downloading {url}")
                payload = response.read()
            text = payload.decode("utf-8-sig")
            rows = list(csv.reader(io.StringIO(text), strict=True))
            if len(rows) < 2:
                raise ValueError("Downloaded CSV must contain a header and at least one row")
            header = [column.strip() for column in rows[0]]
            if len(header) < 2 or not all(header):
                raise ValueError("Downloaded CSV has an invalid header")
            column_count = len(header)
            if any(len(row) != column_count for row in rows[1:]):
                raise ValueError("Downloaded CSV contains rows with inconsistent column counts")
            if not any(any(value.strip() for value in row) for row in rows[1:]):
                raise ValueError("Downloaded CSV contains no data values")

            temporary.write_bytes(payload)
            temporary.replace(destination)
            print(f"CSV downloaded and validated successfully: {destination}")
            return destination
        except (HTTPError, URLError, TimeoutError, OSError, UnicodeDecodeError, csv.Error, RuntimeError, ValueError) as error:
            last_error = error
            temporary.unlink(missing_ok=True)
            if attempt < attempts:
                sleep(2 ** (attempt - 1))

    raise RuntimeError(f"Could not download and validate {url} after {attempts} attempts") from last_error


download_and_validate_csv(
    "https://raw.githubusercontent.com/ali-ce/datasets/master/Most-Expensive-Things/Videogames.csv",
    "./csv_data/video_games.csv",
)

Vamos fazer upload dos dados de video games disponíveis na pasta `csv_data` para o S3.

In [ ]:
def upload_directory(path, bucket_name):
        for root,dirs,files in os.walk(path):
            for file in files:
                file_to_upload = os.path.join(root,file)
                print(f"uploading file {file_to_upload} to {bucket_name}")
                s3_client.upload_file(file_to_upload,bucket_name,file)

upload_directory("csv_data", bucket_name)

Agora iniciamos o ingestion job.

In [ ]:
# ensure that the kb is available
time.sleep(30)
# sync knowledge base
knowledge_base_standard.start_ingestion_job()

Por fim, salvamos o Knowledge Base Id para testar a solução em uma etapa posterior.

In [ ]:
kb_id_standard = knowledge_base_standard.get_knowledge_base_id()

### 2.2 Consultar a Knowledge Base com a API Retrieve and Generate - sem metadata

Vamos testar a knowledge base usando a API [**retrieve_and_generate**](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent-runtime/client/retrieve_and_generate.html). Com essa API, o Bedrock cuida de recuperar as referências necessárias da knowledge base e gerar a resposta final usando um foundation model do Bedrock.

'''
query = "List the video games published by Rockstar Games and released after 2010"
'''

Resultados esperados: Grand Theft Auto V, L.A. Noire, Max Payne 3

In [ ]:
query = "Provide a list of all video games published by Rockstar Games and released after 2010"

In [ ]:
response = bedrock_agent_runtime_client.retrieve_and_generate(
    input={
        "text": query
    },
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            'knowledgeBaseId': kb_id_standard,
            "modelArn": bedrock_model_arn(foundation_model, region),
            "retrievalConfiguration": {
                "vectorSearchConfiguration": {
                    "numberOfResults":5
                } 
            }
        }
    }
)

pprint.pp(response['output']['text'])

#### 2.3 Preparar metadata para ingestão

In [ ]:
import csv
import json

In [ ]:
def generate_json_metadata(csv_file, content_field, metadata_fields, excluded_fields):
    # Open the CSV file and read its contents
    with open(csv_file, 'r') as file:
        reader = csv.DictReader(file)
        headers = reader.fieldnames

    # Create the JSON structure
    json_data = {
        "metadataAttributes": {},
        "documentStructureConfiguration": {
            "type": "RECORD_BASED_STRUCTURE_METADATA",
            "recordBasedStructureMetadata": {
                "contentFields": [
                    {
                        "fieldName": content_field
                    }
                ],
                "metadataFieldsSpecification": {
                    "fieldsToInclude": [],
                    "fieldsToExclude": []
                }
            }
        }
    }

    # Add metadata fields to include
    for field in metadata_fields:
        json_data["documentStructureConfiguration"]["recordBasedStructureMetadata"]["metadataFieldsSpecification"]["fieldsToInclude"].append(
            {
                "fieldName": field
            }
        )

    # Add fields to exclude (all fields not in content_field or metadata_fields)
    if not excluded_fields:
        excluded_fields = set(headers) - set([content_field] + metadata_fields)
    
    for field in excluded_fields:
        json_data["documentStructureConfiguration"]["recordBasedStructureMetadata"]["metadataFieldsSpecification"]["fieldsToExclude"].append(
            {
                "fieldName": field
            }
        )

    # Generate the output JSON file name
    output_file = f"{csv_file.split('.')[0]}.csv.metadata.json"

    # Write the JSON data to the output file
    with open(output_file, 'w') as file:
        json.dump(json_data, file, indent=4)

    print(f"JSON metadata file '{output_file}' has been generated.")

In [ ]:
csv_file = 'csv_data/video_games.csv'
content_field = 'Videogame'
metadata_fields = ['Year', 'Developer', 'Publisher']
excluded_fields =['Description']

generate_json_metadata(csv_file, content_field, metadata_fields, excluded_fields)

In [ ]:
# upload metadata file to S3
upload_directory("csv_data", bucket_name)

# delete metadata file from local
os.remove('csv_data/video_games.csv.metadata.json')

Agora inicie o ingestion job. Como estamos usando os mesmos documentos usados para fixed chunking, estamos pulando a etapa de upload dos documentos para o bucket S3.

In [ ]:
# ensure that the kb is available
time.sleep(30)
# sync knowledge base
knowledge_base_standard.start_ingestion_job()

### 2.4 Consultar a Knowledge Base com a API Retrieve and Generate - sem metadata

Criar o filtro

In [ ]:
one_group_filter= {
    "andAll": [
        {
            "equals": {
                "key": "Publisher",
                "value": "Rockstar Games"
            }
        },
        {
            "greaterThan": {
                "key": "Year",
                "value": 2010
            }
        }
    ]
}

Passe o filtro para `retrievalConfiguration` do [**retrieve_and_generate**](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent-runtime/client/retrieve_and_generate.html).

In [ ]:
response = bedrock_agent_runtime_client.retrieve_and_generate(
    input={
        "text": query
    },
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            'knowledgeBaseId': kb_id_standard,
            "modelArn": bedrock_model_arn(foundation_model, region),
            "retrievalConfiguration": {
                "vectorSearchConfiguration": {
                    "numberOfResults":5,
                    "filter": one_group_filter
                } 
            }
        }
    }
)

print(response['output']['text'])

Como você pode ver, com a API retrieve and generate obtemos a resposta final diretamente. Agora vamos observar as citações da API `RetrieveAndGenerate`. Além disso, vamos observar os chunks recuperados e citações retornadas pelo modelo ao gerar a resposta. Quando fornecemos o contexto relevante ao foundation model junto com a query, ele provavelmente gerará uma resposta de alta qualidade.

In [ ]:
response_standard = workshop_retrieved_references(response)
print("# of citations or chunks used to generate the response: ", len(response_standard))
def citations_rag_print(response_ret):
#structure 'retrievalResults': list of contents. Each list has content, location, score, metadata
    for num,chunk in enumerate(response_ret,1):
        print(f'Chunk {num}: ',chunk['content']['text'],end='\n'*2)
        print(f'Chunk {num} Location: ',chunk['location'],end='\n'*2)
        print(f'Chunk {num} Metadata: ',chunk['metadata'],end='\n'*2)

citations_rag_print(response_standard)

In [ ]:
print("Notebook state is persisted by the sequential runner.")

### Limpeza (Clean up)
Certifique-se de descomentar e executar as células abaixo para excluir os recursos criados neste notebook. Se você planeja executar o notebook `dynamic-metadata-filtering` na seção `03-advanced-concepts`, certifique-se de voltar aqui para excluir os recursos.

In [ ]:
# Cleanup is intentionally deferred to full_cleanup.ipynb.
print("Cleanup deferred to full_cleanup.ipynb.")


In [ ]:
# Cleanup is intentionally deferred to full_cleanup.ipynb.
print("Cleanup deferred to full_cleanup.ipynb.")
